[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA2/blob/main/en/lab3/lab3_part1.ipynb)
# Lab 3: Neural networks with PyTorch
## Part 1. Creating our first neural network model

In this lab we are going to use PyTorch to build and train deep learning models. To begin with, we will see the facilities it provides for creating the different layers of a neural network and how these layers can be combined to build a neural network.

# Prerequisites

## Install packages

For this first part we will only need `numpy`, `torch` (and `pandas`, `sklearn` and `seaborn` to load the dataset).

In [ ]:
import numpy as np
import torch
import seaborn as sns
import pandas as pd

# The Module class in PyTorch

One of the main abstractions in PyTorch is the **`torch.nn.Module`** class.
`Module` allows us to implement neural network layers, encapsulating both **the state** (the parameters of the layer, such as the weight matrix \(\mathbf{W}\) and the *bias* vector \(\mathbf{b}\)) and the **input-output transformation** (the *forward pass*). We are going to create a densely connected layer, that is, a layer where all inputs are connected to all outputs. In addition, and just as was done in *Lab 2*, it will use the *sigmoid* function as the activation function.

The best way to implement our own layer is to extend the `Module` class and implement:

1. **`__init__`**:  
   - The structure of the layer is defined and the tensors that will be the trainable parameters (`torch.nn.Parameter`) are created.  
   - Here you can initialize everything that does not depend on the size of the input, although you can also create parameters that depend on the input.

2. **`forward`**:
   - Here the transformation of the data is defined, using PyTorch operations such as `@` (matmul) or activation functions (`torch.sigmoid`, `torch.relu`, etc.).

## Creating a layer

We are going to create our own layer that inherits from the `Module` class and, therefore, will allow us to use it later in our neural network *model*. When initializing this layer we will only indicate the number of outputs; when building it we will pass the *shape* of the input to it and initialize the parameters randomly (the weights $\mathbf{W}$ and the bias $b$). In the *call* we will do the necessary computations, taking into account that the *matmul* operation allows us to perform the multiplication of matrices and that the activation function is *sigmoid* (to see the activation functions https://docs.pytorch.org/docs/main/nn.html#non-linear-activations-weighted-sum-nonlinearity, we will be using some of them throughout the course).

Design the layer as described. Make the sigmoid output optional.

In [ ]:
import torch.nn as nn

class OurDenseLayer(nn.Module):
    def __init__(self, n_output_nodes, input_dim, no_sigmoid=False):
        super().__init__()
        self.no_sigmoid = not no_sigmoid
        self.n_output_nodes = n_output_nodes
        # Define and initialize parameters: a weight matrix W and a bias b
        # Parameter initialization is random
        self.W = nn.Parameter(torch.randn(input_dim, self.n_output_nodes, dtype=torch.float32), requires_grad=True)
        # TODO: declare the bias
        #self.b = 

    def forward(self, x):
        # Compute z using @
        # TODO: define z 
        # z =...
        # Apply sigmoid if requested
        # TODO: define y 
        #y = ....
        return y

# Concatenating layers to build a network

We are going to create a network, which we will call *model*, using the three layers just as was done in *Lab 2*:

1. The layer $C_0$ has 5 units. It receives the vector $\mathbf{x}$ as input and produces the vector $\mathbf{h_0}$ as output. It has a weight matrix $\mathbf{W_0}$ and a bias vector $\mathbf{b_0}$.

1. The layer $C_1$ has 3 units. It receives the vector $\mathbf{h_0}$ as input and produces the vector $\mathbf{h_1}$ as output. It has a weight matrix $\mathbf{W_1}$ and a bias vector $\mathbf{b_1}$.

1. The layer $C_2$ has 1 unit. It receives the vector $\mathbf{h_1}$ as input and produces the vector $\mathbf{y}$ as output. It has a weight matrix $\mathbf{W_2}$ and a bias vector $\mathbf{b_2}$. **It does not have sigmoid activation.**

In [ ]:
input_size = 10 # The dataset we will use has 10 variables
h0_size = 5
h1_size = 3

class OurNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        # Create the layers
        self.layer0 = OurDenseLayer(h0_size, input_size)
        # TODO - create the other two layers
        

    def forward(self, x):
        # TODO - Call the layers to perform the forward pass

        return y

# Instantiate the model
model = OurNetwork()

## Loading the dataset

We are going to use the same dataset as in *Lab 2*.

In [ ]:
from sklearn.preprocessing import StandardScaler

def load_titanic():
    # Load the Titanic dataset from seaborn
    df = sns.load_dataset('titanic')

    # 1) Select relevant variables and clean
    # Columns we will use
    cols = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
    df = df[cols].copy()

    # Drop rows with missing values
    df = df.dropna(subset=['age', 'embarked', 'fare'])

    # 2) Split labels and features
    y = df['survived'].to_numpy().astype(np.float32)        # labels as float
    X = df.drop(columns=['survived'])

    # 3) One-hot encoding for all categorical variables
    categorical_cols = ['pclass', 'sex', 'embarked', 'alone']
    X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)  # drop_first=True avoids multicollinearity

    # 4) Numeric variables
    numeric_cols = ['age', 'sibsp', 'parch', 'fare']
    X_numeric = X_encoded[numeric_cols + [c for c in X_encoded.columns if c not in numeric_cols]]
    scaler = StandardScaler()
    X_numeric[numeric_cols] = scaler.fit_transform(X_numeric[numeric_cols])

    # 5) Convert to numpy arrays
    X_np = X_numeric.to_numpy().astype(np.float32)
    y_np = y.reshape(-1, 1).astype(np.float32)  # reshape to (n_samples, 1)

    return X_np, y_np

x_data, labels = load_titanic()
x_data = torch.from_numpy(x_data)
labels = torch.from_numpy(labels)

## Training the model

As we did in *Lab 2*, we are going to use PyTorch functions to fit the parameters of the network (now included in our *model*) so that the cost function is minimized. To do so we indicate:

 1. The loss function we want (cross entropy, but using logits since we have not added a sigmoid output to the last layer).
 1. The optimization method to use (gradient descent).

In [ ]:
# TODO - Set the loss function and the gradient-descent algorithm
# Loss (Binary Cross Entropy with logits for better numerical stability)
#loss_fn = 

#optimizer = 

We use the same training loop as in *Lab 2*, but now we do not have the `predict` function and the *VARIABLES* that we had to keep adjusting have not been declared either. What should we use?

In [ ]:
def training_step(x, y):
    # TODO - Complete the next line so it computes the predictions
    #y_pred = 
    
    # Compute the loss with the function chosen above
    loss = loss_fn(y_pred, y)

    # TODO - Complete the rest of the training step
    
    # Accuracy
    errors = torch.abs(y.reshape(-1,1) - torch.sigmoid(y_pred))
    accuracy = torch.sum(1 - errors)
    
    # Return these two values so we can print them when useful
    return (loss, accuracy)

# TRAINING LOOP
num_epochs = 10000
num_samples = x_data.shape[0]

for epoch in range(num_epochs):    
    loss, accuracy = training_step(x_data, labels)
    
    if epoch % 100 == 99:
        print("Epoch:", epoch, 'Loss:', loss.numpy(), 'Accuracy:', accuracy.numpy()/num_samples)


# Exercise

Improve your implementation by making the following changes:
1. Replace the `OurDenseLayer` layers with [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) layers followed by [torch.sigmoid](https://docs.pytorch.org/docs/stable/generated/torch.sigmoid.html) operations.
1. Replace `OurNetwork` by [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html).
1. Change the optimizer to [Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html).